In [1]:
import pandas as pd
from pathlib import Path

# Resolve data dir robustly (works whether kernel cwd is notebooks/python or repo root)
CANDIDATES = [
    Path("../../data/raw/diabetes+130-us+hospitals+for+years+1999-2008"),
    Path("data/raw/diabetes+130-us+hospitals+for+years+1999-2008"),
    Path.cwd() / "data/raw/diabetes+130-us+hospitals+for+years+1999-2008",
    Path.cwd().parent.parent / "data/raw/diabetes+130-us+hospitals+for+years+1999-2008",
]
DATA_DIR = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_DIR is not None, f"Raw data dir not found. Tried: {CANDIDATES}, cwd={Path.cwd()}"
print(f"DATA_DIR: {DATA_DIR.resolve()}")

data_path = DATA_DIR / "diabetic_data.csv"
# NOTE: filename is IDS_mapping.csv (IDS), not IDs_mapping.csv
mapping_path = DATA_DIR / "IDS_mapping.csv"
assert data_path.exists(), f"Missing: {data_path}"
assert mapping_path.exists(), f"Missing: {mapping_path}"

df = pd.read_csv(data_path)
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())
print("\nData types:")
display(df.dtypes)
print("\nBasic info:")
df.info()

DATA_DIR: C:\Users\Administrator\Downloads\tanzania-readmission-intelligence-platform-research-engine\data\raw\diabetes+130-us+hospitals+for+years+1999-2008
Shape: (101766, 50)

Columns:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitt

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO



Data types:


encounter_id                 int64
patient_nbr                  int64
race                        object
gender                      object
age                         object
weight                      object
admission_type_id            int64
discharge_disposition_id     int64
admission_source_id          int64
time_in_hospital             int64
payer_code                  object
medical_specialty           object
num_lab_procedures           int64
num_procedures               int64
num_medications              int64
number_outpatient            int64
number_emergency             int64
number_inpatient             int64
diag_1                      object
diag_2                      object
diag_3                      object
number_diagnoses             int64
max_glu_serum               object
A1Cresult                   object
metformin                   object
repaglinide                 object
nateglinide                 object
chlorpropamide              object
glimepiride         


Basic info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int

In [2]:
# IDs mapping file (admission-type lookup) — was loaded but never used in v1
mapping_df = pd.read_csv(mapping_path)
print("Mapping shape:", mapping_df.shape)
display(mapping_df.head())

# Verse-2 major risk: repeated encounters per patient -> patient-level leakage
print("\nOutcome distribution:")
print(df["readmitted"].value_counts())
print("\n30-day readmission rate:", (df["readmitted"] == "<30").mean())
print("\nUnique patients:", df["patient_nbr"].nunique(), "vs rows:", len(df))
print("\nTop encounter counts per patient:")
display(df["patient_nbr"].value_counts().head(10))
print("Duplicated encounter_id:", df["encounter_id"].duplicated().sum())

Mapping shape: (67, 2)


,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available



Outcome distribution:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

30-day readmission rate: 0.11159915885462728

Unique patients: 71518 vs rows: 101766

Top encounter counts per patient:


patient_nbr
88785891    40
43140906    28
1660293     23
88227540    23
23199021    23
23643405    22
84428613    22
92709351    21
88789707    20
29903877    20
Name: count, dtype: int64

Duplicated encounter_id: 0
